# **K-NN and SVM: explanation through proximity**

### Practice for the “K-NN” and “SVM” lessons of the module “Nonlinear interpretable models”

The lessons made several claims worth checking on data rather than taking on trust:

1. the explanation of a K-NN prediction is the neighbors themselves plus the rule that picked them: the metric and the number $k$;
2. changing the scale or the metric changes the neighbors, and with them the answer to “why”;
3. the weights of a linear SVM set the position of the hyperplane and cannot be read as feature importances;
4. the SVM decision depends only on the support vectors, and a kernel SVM is still explained through them;
5. in high dimension the nearest and the farthest neighbor become almost indistinguishable (Beyer et al., 1999).

The data are `diabetes.csv` (Pima Indians Diabetes, 768 patients), features `Glucose`, `BMI` and `Age`, target `Outcome` — whether the patient has diabetes. The object we explain is patient $a$ in row 15 of the file; in the lesson “K-NN”, the step “Scaling the data” mentions the same patient as an example of the nearest neighbor changing after standardization.

All computations are deterministic: your numbers will match those in the exercises. Enjoy!

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42
FEATURES = ['Glucose', 'BMI', 'Age']
PATIENT = 15                      # row of the file, counting from zero: object a of the lesson “K-NN”

df = pd.read_csv('https://github.com/SadSabrina/open-xai-materials/raw/refs/heads/main/data/diabetes.csv')
X, y = df[FEATURES], df['Outcome']
print(df.shape)
df.loc[[15, 18, 23], FEATURES + ['Outcome']]

## Data

The three objects: $a$ (row 15), $b$ (row 18) and $c$ (row 23).

## Part 1. K-NN: the explanation is the neighbors

The function `neighbours` returns the five nearest neighbors of an object among the other 767 rows: row number, distance and the neighbor's `Outcome`. The prediction of K-NN with $k=5$ is the majority vote of these five, and the share of neighbors with `Outcome = 1` is its class-1 probability. Below are the neighbors of patient $a$ on the raw, unscaled features.

In [ ]:
def neighbours(Xs, i, k=5, p=2):
    """k nearest neighbors of object i among all other rows of Xs (Minkowski metric with parameter p)."""
    rest = Xs.drop(index=i)
    dist, idx = NearestNeighbors(n_neighbors=k, p=p).fit(rest).kneighbors(Xs.loc[[i]])
    return pd.DataFrame({'row': rest.index[idx[0]], 'distance': dist[0].round(3),
                         'Outcome': y.loc[rest.index[idx[0]]].values})

raw = neighbours(X, PATIENT)
raw

## Part 2. SVM: support vectors

We split the data into train and test (25%, stratified), standardize the features on the training part and fit a linear SVM with $C=1$. After fitting, `SVC` has `support_` (indices of the support vectors in the training set), `n_support_` (their number per class), `dual_coef_` (the products $\alpha_i y_i$) and `intercept_` (the bias $b$).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)

svm = SVC(kernel='linear', C=1).fit(Z_tr, y_tr)
print('support vectors per class:', svm.n_support_, ' total:', svm.n_support_.sum(), 'of', len(Z_tr))
print('test accuracy:', round(svm.score(Z_te, y_te), 3))
print('w =', dict(zip(FEATURES, svm.coef_[0].round(3))))

## Tasks

1. **Neighbors after standardization.** Standardize all three features over all 768 rows ($z = (x-\mu)/\sigma$, `StandardScaler`) and find the five neighbors of patient $a$. How many of them have `Outcome = 1`? What does K-NN with $k = 5$ predict, and how many neighbors coincide with those on the raw features?
2. **Changing the metric.** On the standardized features, replace the Euclidean metric ($p = 2$) with the Manhattan one ($p = 1$). How many of the five neighbors of patient $a$ stay the same?
3. **Scale and quality.** Compare the test accuracy of K-NN at $k = 1, 5, 15, 25$ on the raw and on the standardized features (scaling fitted on the training part).
4. **How many objects hold the decision.** How many support vectors does the linear SVM with $C=1$ above have? How does their number change at $C = 0.01$, $0.1$ and $10$? Check the claim of the lesson “SVM”: refit the SVM on the support vectors only and compare $w$ and $b$.
5. **SVM weights and scale.** Fit the same linear SVM on the raw, unstandardized features and compare the `coef_` of the two models. Is the order of the features by absolute weight the same?
6. **A kernel SVM through its support vectors.** Fit an SVM with an RBF kernel (`C=1`, `gamma='scale'`) on the standardized features of task 1, on all rows except patient $a$. Decompose the `decision_function` of patient $a$ into the contributions of the support vectors $\alpha_i y_i K(x_i, x)$ and check that their sum plus $b$ equals `decision_function`. Whose contributions are the largest in absolute value, and what share of the sum of absolute contributions do the ten largest give?
7. **The curse of dimensionality.** Append $d = 0, 10, 100, 1000$ noise features drawn from $\mathcal{N}(0, 1)$ to the three standardized features (`np.random.default_rng(0)`, one generator for all $d$ in this order) and compute for patient $a$ the ratio $D_{\max}/D_{\min}$ — the distance to the farthest and to the nearest object. What happens to it as $d$ grows?

## What to take away

The explanation of K-NN and SVM consists of objects, not weights: neighbors for K-NN and support vectors for SVM. But both explanations belong not only to the model, but also to the choice of scale, metric and kernel: the same object on the same data gets other neighbors, and a linear SVM gets another order of features by weight, once the scale changes. So together with the explanation one names the space and the measure in which proximity was computed.

Proximity also has a limit: as the number of features unrelated to the task grows, the farthest object comes close to the nearest one, and a “neighbor” stops meaning anything. Explaining through proximity works while there are few features and they are meaningful.